# DS6 — Dynamical Systems (DyS) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral)
**Model family:** Dynamical Systems (DyS)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_dys_blueprint.md`

---

## Scope and purpose

Fourth DS6 CAS model. Depends on StoP (73, mu_hat/sig_hat) and InT (75, H_emp), both
complete. Takes the sigma->0 limit of StoP's SDE to classify each participant-period into a
deterministic-regime label, then examines regime transitions across the baseline-intervention
boundary and bridges to InT's entropy.

- **Goal:** Classify DS6 participants by deterministic vs stochastic regime, per period;
  characterize regime transitions; validate against InT's H_emp.
- **Not goal:** Phenotype classification (Step 2), personalization, cross-dataset harmonization.
- **Output:** Per-participant regime labels (both periods), regime transition matrix,
  InT bridge validation, condition x period x regime summary.

### FLAG decisions in effect (blueprint Part 2)

| Flag | Decision |
|---|---|
| FLAG DyS-1 | Per-period sigma threshold (median sig_hat within that period), NOT one pooled threshold -- controls for StoP's own documented window-length precision asymmetry |
| FLAG DyS-2 | 3-way asymmetric regime structure (declining/growing/resting) for the deterministic-eligible group, reusing StoP's MU_THRESHOLD=0.5 |
| FLAG DyS-3 | Regime-transition contingency table (baseline x intervention) -- DS6-specific, no DS5 equivalent |

### WARNING -- why this is not a DS5 port (structural, not stylistic)

DS5's DyS took the sigma->0 limit of a state variable bounded on BOTH sides ([0,1] override
rate), so a nonzero drift drives the trajectory to a **boundary** fixed point on either end
-- hence DS5's careful "predicted_boundary_fp" terminology (never "attractor").

DS6's state variable (StoP's daily attempt count N_t) is bounded ONLY below (at 0 -- can't
have negative attempts) and has NO ceiling. The same sigma->0 limit therefore gives a
structurally ASYMMETRIC picture DS5 has no equivalent for:
- mu_hat << 0: deterministic drift toward the floor at N=0 (a real boundary, DS6's one-sided
  analog of DS5's boundary fp)
- mu_hat >> 0: deterministic drift upward with NO ceiling -- unbounded growth, no boundary
  to reach. DS5's double-bounded case cannot produce this regime.
- mu_hat ~ 0: hovers near an interior resting level (mean_N) -- a genuine classical fixed
  point (dN/dt=0), not a boundary approximation.

### Proxy warning

Regime labels describe patterns in a behavioral-volume proxy (StoP's N_t), not directly
observed psychological states (habituation, disengagement, self-control). "Declining" means
declining ATTEMPT COUNT, not necessarily improved self-regulation.

---

## Section 1: Pure Theory -- Dynamical Systems in CAS (paper Section 2.2)

> "State-space variables: Possible variables include attempt volume, opening rate, dismissal
> rate, target-app concentration, time since prior attempt, repeated-override rate, and
> condition exposure."
>
> "System stability: The model asks whether users fall into stable response attractors,
> oscillate between regimes, rebound, or drift toward disengagement."

DS6 cannot use "dismissal rate"/"repeated-override rate" (no outcome variance). Uses the
paper's own first-listed variable: **attempt volume** (StoP's N_t).

### The sigma->0 reduction

    Full SDE (StoP, notebook 73):     dN_t = mu*dt + sigma*dW_t
    Deterministic limit (sigma->0):    dN/dt = mu   (constant drift ODE)

Since N_t >= 0 with no upper bound (unlike DS5's [0,1]), the deterministic trajectory's fate
depends on the SIGN of mu, not just its magnitude relative to a symmetric bound:

    mu < -MU_THRESHOLD:  N(t) -> 0 (floor)          -- "declining"
    mu > +MU_THRESHOLD:  N(t) -> infinity (no ceiling) -- "growing"
    |mu| <= MU_THRESHOLD: N(t) ~ mean_N (interior fp) -- "resting"
    sigma >= threshold:   deterministic limit not a good description -- "stochastic"

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT     = Path('..').resolve()
STOP_DIR = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'stochastic_processes'
INT_DIR  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'information_theory'
REPORTS  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'dynamical_systems'
FIGURES  = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MU_THRESHOLD = 0.5   # reused from StoP (FLAG 4 there)

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('REPORTS :', REPORTS)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load StoP parameters
# =============================================================================
stop_params = pd.read_csv(STOP_DIR / 'participant_stop_parameters.csv')
print(f'StoP participants: {len(stop_params)}')
print(stop_params[['mu_hat_baseline', 'sig_hat_baseline', 'mu_hat_intervention', 'sig_hat_intervention']].describe().round(3))

---

## Section 2: Regime Classification (FLAG DyS-1, DyS-2)

In [ ]:
# =============================================================================
# Per-period sigma threshold (median among eligible, i.e. non-NaN, participants)
# =============================================================================
sigma_thresholds = {}
for period in ['baseline', 'intervention']:
    col = f'sig_hat_{period}'
    thr = stop_params[col].median()
    sigma_thresholds[period] = thr
    print(f'{period}: sigma_threshold = {thr:.4f} (median sig_hat)')

def classify_regime(mu, sig, sigma_thr):
    if pd.isna(mu) or pd.isna(sig):
        return np.nan
    if sig >= sigma_thr:
        return 'stochastic'
    if mu < -MU_THRESHOLD:
        return 'declining'
    if mu > MU_THRESHOLD:
        return 'growing'
    return 'resting'

regime_df = stop_params[['participant_id_canonical', 'study_condition']].copy()
for period in ['baseline', 'intervention']:
    thr = sigma_thresholds[period]
    regime_df[f'regime_{period}'] = [
        classify_regime(mu, sig, thr)
        for mu, sig in zip(stop_params[f'mu_hat_{period}'], stop_params[f'sig_hat_{period}'])
    ]
    regime_df[f'is_dys_eligible_{period}'] = stop_params[f'sig_hat_{period}'] < thr
    regime_df[f'sigma_threshold_{period}'] = thr
    regime_df[f'mu_hat_{period}'] = stop_params[f'mu_hat_{period}']
    regime_df[f'sig_hat_{period}'] = stop_params[f'sig_hat_{period}']

for period in ['baseline', 'intervention']:
    print(f'\n{period} regime distribution:')
    print(regime_df[f'regime_{period}'].value_counts())

---

## Section 3: Regime Transitions Across the Baseline-Intervention Boundary (FLAG DyS-3)

In [ ]:
# =============================================================================
# 4x4 regime transition contingency table
# =============================================================================
both_elig = regime_df.dropna(subset=['regime_baseline', 'regime_intervention'])
print(f'Participants with valid regime in BOTH periods: {len(both_elig)}')

regime_transition_matrix = pd.crosstab(
    both_elig['regime_baseline'], both_elig['regime_intervention'],
    rownames=['regime_baseline'], colnames=['regime_intervention']
)
print('\nRegime transition counts (rows=baseline, cols=intervention):')
print(regime_transition_matrix)

n_same = sum(both_elig['regime_baseline'] == both_elig['regime_intervention'])
print(f'\nParticipants staying in the SAME regime: {n_same} / {len(both_elig)} ({100*n_same/len(both_elig):.1f}%)')

regime_transition_matrix_out = regime_transition_matrix.reset_index()

---

## Section 4: Bridge to InT -- Regime vs H_emp (FLAG: prediction stated before running)

In [ ]:
# =============================================================================
# Prediction: deterministic-eligible participants should show LOWER H_emp than
# stochastic-regime participants (per period). Tested, not assumed.
# =============================================================================
entropy_df = pd.read_csv(INT_DIR / 'participant_entropy_metrics.csv')
bridge_df = regime_df.merge(
    entropy_df[['participant_id_canonical', 'H_emp_baseline', 'H_emp_intervention']],
    on='participant_id_canonical', how='left'
)

regime_int_rows = []
for period in ['baseline', 'intervention']:
    det = bridge_df.loc[bridge_df[f'is_dys_eligible_{period}'] == True, f'H_emp_{period}'].dropna()
    stoch = bridge_df.loc[bridge_df[f'is_dys_eligible_{period}'] == False, f'H_emp_{period}'].dropna()
    if len(det) > 1 and len(stoch) > 1:
        t_stat, p_val = stats.ttest_ind(det, stoch, equal_var=False)
    else:
        t_stat, p_val = np.nan, np.nan
    regime_int_rows.append(dict(
        study_period=period, n_deterministic=len(det), n_stochastic=len(stoch),
        H_emp_mean_deterministic=det.mean(), H_emp_mean_stochastic=stoch.mean(),
        t_stat=t_stat, p_value=p_val,
    ))
    print(f'{period}: H_emp deterministic mean={det.mean():.3f} (n={len(det)}), '
          f'stochastic mean={stoch.mean():.3f} (n={len(stoch)}), '
          f't={t_stat:.3f}, p={p_val:.4g}')

regime_int_bridge = pd.DataFrame(regime_int_rows).round(4)

---

## Section 5: Condition x Period x Regime Summary

In [ ]:
# =============================================================================
# Condition x period x regime counts
# =============================================================================
cond_regime_rows = []
for cond in ['control', 'default', 'intervention']:
    for period in ['baseline', 'intervention']:
        sub = regime_df[regime_df['study_condition'] == cond]
        counts = sub[f'regime_{period}'].value_counts()
        for regime_label in ['declining', 'growing', 'resting', 'stochastic']:
            cond_regime_rows.append(dict(
                study_condition=cond, study_period=period, regime=regime_label,
                n=int(counts.get(regime_label, 0))
            ))
condition_regime_summary = pd.DataFrame(cond_regime_rows)
print(condition_regime_summary.pivot_table(index=['study_condition','study_period'], columns='regime', values='n').to_string())

In [ ]:
# =============================================================================
# Figure 1: regime distribution by period
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, period in zip(axes, ['baseline', 'intervention']):
    counts = regime_df[f'regime_{period}'].value_counts().reindex(
        ['declining', 'resting', 'growing', 'stochastic'])
    ax.bar(counts.index, counts.values, color=['#C44E52','#55A868','#4C72B0','#8172B2'])
    ax.set_title(f'DS6 DyS -- regime distribution ({period})')
    ax.set_ylabel('Participants')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_dys_regime_distribution.png', dpi=150)
plt.show()
print('Saved fig1_dys_regime_distribution.png')

In [ ]:
# =============================================================================
# Figure 2: regime transition heatmap
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 6))
order = ['declining', 'resting', 'growing', 'stochastic']
mat = regime_transition_matrix.reindex(index=order, columns=order, fill_value=0)
im = ax.imshow(mat.values, cmap='viridis')
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel('intervention regime'); ax.set_ylabel('baseline regime')
for i in range(len(order)):
    for j in range(len(order)):
        ax.text(j, i, mat.values[i,j], ha='center', va='center', color='white', fontsize=10)
ax.set_title('DS6 DyS -- regime transition counts (baseline -> intervention)')
plt.colorbar(im, ax=ax, label='count')
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_dys_regime_transition_heatmap.png', dpi=150)
plt.show()
print('Saved fig2_dys_regime_transition_heatmap.png')

In [ ]:
# =============================================================================
# Figure 3: regime vs H_emp bridge
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, period in zip(axes, ['baseline', 'intervention']):
    data = [
        bridge_df.loc[bridge_df[f'is_dys_eligible_{period}']==True, f'H_emp_{period}'].dropna(),
        bridge_df.loc[bridge_df[f'is_dys_eligible_{period}']==False, f'H_emp_{period}'].dropna(),
    ]
    ax.violinplot(data, showmeans=True)
    ax.set_xticks([1,2]); ax.set_xticklabels(['deterministic', 'stochastic'])
    ax.set_ylabel('H_emp (InT, notebook 75)')
    ax.set_title(f'{period}')
plt.suptitle('DS6 DyS-InT bridge: H_emp by regime eligibility')
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_dys_regime_hemp_bridge.png', dpi=150)
plt.show()
print('Saved fig3_dys_regime_hemp_bridge.png')

In [ ]:
# =============================================================================
# Figure 4: mu vs sigma scatter, colored by regime
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {'declining': '#C44E52', 'resting': '#55A868', 'growing': '#4C72B0', 'stochastic': '#8172B2'}
for ax, period in zip(axes, ['baseline', 'intervention']):
    for regime_label, color in colors.items():
        sub = regime_df[regime_df[f'regime_{period}'] == regime_label]
        ax.scatter(sub[f'mu_hat_{period}'], sub[f'sig_hat_{period}'], c=color, label=regime_label, alpha=0.6, s=20)
    ax.axhline(sigma_thresholds[period], color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(-MU_THRESHOLD, color='gray', linewidth=0.5, linestyle=':')
    ax.axvline(MU_THRESHOLD, color='gray', linewidth=0.5, linestyle=':')
    ax.set_xlabel('mu_hat'); ax.set_ylabel('sig_hat')
    ax.set_title(f'{period}')
    ax.legend(fontsize=7)
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_dys_mu_sigma_regime_scatter.png', dpi=150)
plt.show()
print('Saved fig4_dys_mu_sigma_regime_scatter.png')

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
out_cols = ['participant_id_canonical', 'study_condition',
            'regime_baseline', 'is_dys_eligible_baseline', 'sigma_threshold_baseline',
            'mu_hat_baseline', 'sig_hat_baseline',
            'regime_intervention', 'is_dys_eligible_intervention', 'sigma_threshold_intervention',
            'mu_hat_intervention', 'sig_hat_intervention']
participant_regime_classification = regime_df[out_cols].copy()
participant_regime_classification.to_csv(REPORTS / 'participant_regime_classification.csv', index=False)
print(f'Saved participant_regime_classification.csv ({len(participant_regime_classification)} rows)')

regime_transition_matrix_out.to_csv(REPORTS / 'regime_transition_matrix.csv', index=False)
print(f'Saved regime_transition_matrix.csv ({len(regime_transition_matrix_out)} rows)')

regime_int_bridge.to_csv(REPORTS / 'regime_int_bridge.csv', index=False)
print(f'Saved regime_int_bridge.csv ({len(regime_int_bridge)} rows)')

condition_regime_summary.to_csv(REPORTS / 'condition_regime_summary.csv', index=False)
print(f'Saved condition_regime_summary.csv ({len(condition_regime_summary)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
summary_md = f"""# DS6 DyS -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/72_ds6_dynamical_systems.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_dys_blueprint.md

## Sigma thresholds (FLAG DyS-1, per-period)
- Baseline: {sigma_thresholds['baseline']:.4f}
- Intervention: {sigma_thresholds['intervention']:.4f}

## Regime distributions
Baseline:
{regime_df['regime_baseline'].value_counts().to_string()}

Intervention:
{regime_df['regime_intervention'].value_counts().to_string()}

## Regime transitions
Participants with valid regime both periods: {len(both_elig)}
Staying in the SAME regime: {n_same} ({100*n_same/len(both_elig):.1f}%)

## InT bridge (regime vs H_emp)
{regime_int_bridge.to_string(index=False)}

## Proxy warning
Regime labels describe patterns in StoP's daily-attempt-count proxy, not directly observed
psychological states (habituation, disengagement, self-control).

## Known limitations
- Deterministic-limit interpretation (sigma->0) is an approximation; DS6's actual process
  has sigma > 0 throughout -- same caveat DS5's DyS notebook carried (FLAG A there).
- Per-period sigma threshold means "deterministic" is relative to that period's own
  distribution, not an absolute cross-period standard -- do not compare eligibility
  proportions across periods as if using the same cutoff value.
- study_condition is quasi-experimental (cohort-level), not individually randomized.
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 6: Interpretation and Limits

### Headline finding: regime shift independently confirms StoP's "level moved, trend didn't"

Baseline "declining" (30) nearly vanishes in intervention (2) -- but intervention is
dominated by "resting" (95, up from 51), not decline. This independently confirms StoP's
(73) headline finding via a completely different method: StoP found the mean daily-volume
LEVEL drops (45.1->29.1 events/day) but the within-period drift TREND (h_period) does not
differ significantly. DyS shows the same story from a regime-classification angle:
deterministic participants settle into a flat trend during intervention, consistent with a
level shift BETWEEN periods rather than a declining trend WITHIN the intervention period.

### Interpretive caution: mu_hat dispersion differs by window length

std(mu_hat_baseline)=3.04 vs std(mu_hat_intervention)=0.96. Since MU_THRESHOLD=0.5 is
absolute and the intervention window is ~2x baseline's, part of the "resting" surge may
reflect narrower estimation noise with more data, not purely a behavioral shift. Flagged,
not resolved -- same class of caution already recorded for StoP's sig_hat/Fano comparison.

### What the DyS model reveals

| Result | Interpretation |
|---|---|
| Regime distribution per period | Share of participants whose daily-count trajectory is well-described by a deterministic drift vs a noise-dominated process |
| Regime transitions | Whether participants stay in the same dynamical regime across the baseline-intervention boundary, or shift |
| Regime vs H_emp bridge | Whether the deterministic/stochastic classification is validated by an independent entropy measure |

### What this model does NOT show

- A causal account of why a participant is in a given regime.
- "Growing"/"declining" as psychological habituation or self-control -- only attempt-count trend.
- Cross-period eligibility-share comparability (per-period thresholds, FLAG DyS-1).

### Forward connections (informational only -- not implemented here)

- **DyS -> Step 2 phenotyping:** regime labels (both periods) and the transition matrix are
  candidate phenotype dimensions once DS6 Step 1 progresses further (SyD, StM remain).
- **DyS -> SyD:** a participant's regime trajectory could inform stock/flow framing if SyD
  is built for DS6 next.